# 08. Recommender Evaluation and Ranking

This notebook evaluates the anime recommendation engine as a product ranking system. The task is not to predict exact ratings; it is to rank plausible unseen anime for a user profile and test whether a held-out liked title appears near the top.

The current evaluator uses the 2026 public-list interaction layer when `data/processed/current_user_ratings.csv` exists. Those rows include scored catalog-matched anime across completed, watching, on-hold, dropped, and plan-to-watch statuses. Positive seeds are ratings `>= 7`, and dropped titles are explicitly excluded from positive training evidence.

The comparison now goes beyond the assignment-style baseline/SVD setup. It tests recommenders built from the data we gathered: popularity, AniList/MAL metadata, relation and recommendation graphs, voice actors/staff, collaborative SVD, and blended product hybrids.

For the final run, the evaluator is configured to use every eligible user after the 50k model-user cap. The advanced rerankers use separate train/evaluation case budgets; the hybrid weight search acts as the validation step for product blending, while the reported leaderboard is read from held-out evaluation cases.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_colwidth', 180)

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

def rel(path):
    text = str(path)
    if not any(token in text for token in ['\\', '/', ':']):
        return text
    try:
        candidate = Path(text)
        if not candidate.is_absolute():
            return text.replace('/', '\\')
        return str(candidate.resolve().relative_to(BASE_DIR))
    except Exception:
        return text.replace(str(BASE_DIR), '').lstrip('\\/').replace('/', '\\')

def sanitize_output_line(line):
    text = str(line).replace(str(BASE_DIR), '').lstrip('\\/')
    text = text.replace(str(Path(sys.executable)), 'python')
    return text

ARTIFACT_DIR = BASE_DIR / 'artifacts' / 'recommendation'
PLOT_DIR = BASE_DIR / 'artifacts' / 'plots' / 'recommender'
SCRIPT_PATH = BASE_DIR / 'src' / '08_evaluate_recommenders.py'

SUMMARY_PATH = ARTIFACT_DIR / 'recommendation_summary.json'
METRICS_PATH = ARTIFACT_DIR / 'evaluation_metrics.csv'
LEVEL_METRICS_PATH = ARTIFACT_DIR / 'metrics_by_user_level.csv'
USER_EVAL_PATH = ARTIFACT_DIR / 'user_level_eval_sample.csv'
EXAMPLES_PATH = ARTIFACT_DIR / 'recommendation_examples.csv'
ALIGNMENT_PATH = ARTIFACT_DIR / 'data_alignment.csv'
BEGINNER_CANDIDATES_PATH = ARTIFACT_DIR / 'beginner_entrypoint_candidates.csv'
HYBRID_COMPONENT_SEARCH_PATH = ARTIFACT_DIR / 'hybrid_component_weight_search.csv'
LEVEL_HYBRID_COMPONENT_SEARCH_PATH = ARTIFACT_DIR / 'level_hybrid_component_weight_search.csv'
ADVANCED_METRICS_PATH = ARTIFACT_DIR / 'advanced_ranker_metrics.csv'
ADVANCED_LEVEL_METRICS_PATH = ARTIFACT_DIR / 'advanced_ranker_metrics_by_user_level.csv'
ADVANCED_INVENTORY_PATH = ARTIFACT_DIR / 'advanced_architecture_inventory.csv'
ADVANCED_TRAINING_LOG_PATH = ARTIFACT_DIR / 'advanced_training_log.csv'
CLASSICAL_TRAINING_LOG_PATH = ARTIFACT_DIR / 'classical_training_log.csv'
ADVANCED_MODEL_CONFIG_PATH = ARTIFACT_DIR / 'advanced_model_configs.csv'
ADVANCED_ERROR_CASES_PATH = ARTIFACT_DIR / 'advanced_systematic_error_cases.csv'

# Main run controls. Set RUN_PIPELINE=True when you want to retrain/re-evaluate.
RUN_PIPELINE = True
MIN_ITEM_POSITIVES = 10        # Candidate pool threshold.
MAX_MODEL_USERS = 50_000
NEGATIVES_PER_USER = 200
ADVANCED_TRAIN_CASES = 25_000
ADVANCED_EVAL_CASES = 25_000
# Final run mode: use every eligible evaluation user after MAX_MODEL_USERS.
# If the laptop becomes unstable, set USE_ALL_EVAL_USERS=False for a capped smoke run.
USE_ALL_EVAL_USERS = True
MAX_EVAL_USERS = None if USE_ALL_EVAL_USERS else ADVANCED_EVAL_CASES
PROGRESS_INTERVAL = 2_500
RUN_ADVANCED = True

command = [
    sys.executable,
    str(SCRIPT_PATH),
    '--min-item-positives', str(MIN_ITEM_POSITIVES),
    '--max-model-users', str(MAX_MODEL_USERS),
    '--negatives-per-user', str(NEGATIVES_PER_USER),
    '--progress-interval', str(PROGRESS_INTERVAL),
    '--advanced-train-cases', str(ADVANCED_TRAIN_CASES),
    '--advanced-eval-cases', str(ADVANCED_EVAL_CASES),
]
if MAX_EVAL_USERS is not None:
    command += ['--max-eval-users', str(MAX_EVAL_USERS)]
if not RUN_ADVANCED:
    command += ['--skip-advanced']

def run_with_live_progress(cmd):
    printable = [cmd[0], rel(cmd[1]), *cmd[2:]]
    print('Running:', ' '.join(printable), flush=True)
    print('Progress is streamed from the evaluator: stage bars show the full pipeline; model lines show the current trained ranker.\n', flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=BASE_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(sanitize_output_line(line.rstrip()), flush=True)
    return_code = process.wait()
    elapsed = time.perf_counter() - started
    print(f'Pipeline wall time: {elapsed/60:.1f} minutes ({elapsed:.1f}s)', flush=True)
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)

needed = [METRICS_PATH, LEVEL_METRICS_PATH, ADVANCED_METRICS_PATH, ADVANCED_TRAINING_LOG_PATH]
if RUN_PIPELINE or any(not path.exists() for path in needed):
    run_with_live_progress(command)

summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
def rel_payload(value):
    if isinstance(value, dict):
        return {key: rel_payload(item) for key, item in value.items()}
    if isinstance(value, list):
        return [rel_payload(item) for item in value]
    if isinstance(value, str):
        return rel(value)
    return value
summary = rel_payload(summary)
metrics = pd.read_csv(METRICS_PATH)
level_metrics = pd.read_csv(LEVEL_METRICS_PATH)
user_eval = pd.read_csv(USER_EVAL_PATH)
examples = pd.read_csv(EXAMPLES_PATH)
alignment = pd.read_csv(ALIGNMENT_PATH)
beginner_candidates = pd.read_csv(BEGINNER_CANDIDATES_PATH)
hybrid_component_search = pd.read_csv(HYBRID_COMPONENT_SEARCH_PATH) if HYBRID_COMPONENT_SEARCH_PATH.exists() else pd.DataFrame()
level_hybrid_component_search = pd.read_csv(LEVEL_HYBRID_COMPONENT_SEARCH_PATH) if LEVEL_HYBRID_COMPONENT_SEARCH_PATH.exists() else pd.DataFrame()
advanced_metrics = pd.read_csv(ADVANCED_METRICS_PATH) if ADVANCED_METRICS_PATH.exists() else pd.DataFrame()
advanced_level_metrics = pd.read_csv(ADVANCED_LEVEL_METRICS_PATH) if ADVANCED_LEVEL_METRICS_PATH.exists() else pd.DataFrame()
advanced_inventory = pd.read_csv(ADVANCED_INVENTORY_PATH) if ADVANCED_INVENTORY_PATH.exists() else pd.DataFrame()
advanced_training_log = pd.read_csv(ADVANCED_TRAINING_LOG_PATH) if ADVANCED_TRAINING_LOG_PATH.exists() else pd.DataFrame()
classical_training_log = pd.read_csv(CLASSICAL_TRAINING_LOG_PATH) if CLASSICAL_TRAINING_LOG_PATH.exists() else pd.DataFrame()
advanced_model_configs = pd.read_csv(ADVANCED_MODEL_CONFIG_PATH) if ADVANCED_MODEL_CONFIG_PATH.exists() else pd.DataFrame()
advanced_error_cases = pd.read_csv(ADVANCED_ERROR_CASES_PATH) if ADVANCED_ERROR_CASES_PATH.exists() else pd.DataFrame()

overview = pd.DataFrame([
    {'item': 'catalog source', 'value': summary.get('catalog_source')},
    {'item': 'ratings source', 'value': summary.get('ratings_source')},
    {'item': 'candidate pool', 'value': f"{summary['candidate_pool']['items']:,} anime with >= {MIN_ITEM_POSITIVES} positives"},
    {'item': 'model users', 'value': f"{summary.get('model_users', 0):,}"},
    {'item': 'evaluated users', 'value': f"{summary.get('evaluated_users', 0):,}"},
    {'item': 'positive interactions', 'value': f"{summary.get('positive_interactions', 0):,}"},
    {'item': 'advanced train/eval cases', 'value': f"{ADVANCED_TRAIN_CASES:,} / {ADVANCED_EVAL_CASES:,}"},
])
overview



## Product Framing

The final product has one true cold-start path plus four evaluated user profile bands:

- **Cold Starter (0 known anime)**: ask for favorite genres/tags, then ask whether the user recognizes popular entry titles. This path is not included in offline ranking plots because it has no interaction profile.
- **Beginner (1-49 known anime)**: favor safe, popular, high-score, lower-commitment entry points.
- **Casual (50-149 known anime)**: use similar anime, direct relations, and popularity anchors because the profile is usable but still small.
- **Fan (150-499 known anime)**: collaborative ranking becomes strong; add relation continuation, current shows, genre expansion, and people/staff signals.
- **Veteran (500+ known anime)**: prioritize novelty, long-tail discovery, graph expansion, and people/staff paths because obvious titles are often already known.

This is why the product should not output one flat list. The target UI is closer to streaming rows: general recommendations, because-you-liked rows, continuation rows, VA/staff rows, and controlled exploration rows.


In [ ]:
pd.DataFrame(summary['user_levels'])

## User Controls

The recommender should not behave like a black box. Users should be able to control broad filters before ranking: genre, demographic/content rating, explicit content, score floor, length, and release year. These controls are product constraints applied before or during ranking, not replacements for the ranking model.

In [ ]:
pd.DataFrame([{'control': key, 'meaning': value} for key, value in summary['control_filters'].items()])

## Data Alignment

The recommendation model aligns four layers:

- `anime_dataset.csv`: one row per recommendable anime catalog entry.
- `current_user_ratings.csv`: current public-list interaction rows, using scored catalog-matched anime.
- `anime_voice_actor_edges.csv` and `anime_staff_edges.csv`: people signals for VA/director/creator recommendation rows.
- catalog relation/recommendation fields: graph structure for franchise navigation and similar-anime discovery.

A positive collaborative interaction is `rating >= 7`, matching MAL's `Good` label. Dropped rows are not positive seeds even if they have a score. The candidate pool is restricted to catalog anime with enough positive interaction evidence, while product guardrails and row generation can still use metadata, graph, and people signals outside pure matrix factorization.


In [ ]:
alignment

## Evaluation Protocol

This is a ranking evaluation over the full eligible current-rating user set, not a 15k-user cap. For each eligible user, one liked anime is deterministically held out. The model receives the remaining liked anime as the user's profile, and the held-out user-item pair is removed from the training matrix.

Each method ranks the held-out liked title against sampled candidate negatives. The evaluation also writes harder sensitivity artifacts with larger negative pools and a full-catalog sample. Because public MAL lists do not provide reliable event timestamps, this is not chronological next-watch prediction; it is controlled liked-title recovery.


## Systems Compared

The evaluation is now split into three layers so the model choice is defensible instead of just reporting one large leaderboard.

### Baselines
- `popularity_baseline`: non-personalized crowd baseline. This is a strong anime baseline because popular, high-visibility shows are genuinely useful recommendations for many users.

### Classical and Product Signals
- `metadata_content`: profile similarity from genres, tags, explicit tags, demographics, studios, source, rating, and type.
- `graph_related`: relation and recommendation edges from the user's liked titles.
- `people_staff_affinity`: shared voice actors plus director/original creator/original story signals.
- `item_knn_collaborative`: learned item-item neighborhood similarity in the collaborative latent space.
- `latent_svd`: collaborative filtering from the user-anime positive interaction matrix.
- `full_product_hybrid`: fixed hand-designed blend of SVD, popularity, metadata, graph, and people/staff signals.
- `tuned_product_hybrid`: global blend selected by a component-weight search.
- `level_tuned_product_hybrid`: profile-band-specific tuned blend; this is the current product-safe general row model.

### Advanced Architectures
- `pointwise_logistic_reranker`, `gbdt_signal_reranker`, `lightgbm_lambdarank`, `xgboost_pairwise_ranker`, `catboost_yetirank`, and `neural_mlp_reranker`: learned rerankers trained on candidate-level product features and evaluated on a separate held-out user split.
- `torch_feature_mlp_reranker`: a neural feature reranker over the same candidate features.
- `implicit_als_cf` and `torch_two_tower_cf`: deeper collaborative architectures trained from the sparse user-anime interaction matrix.

The goal is to choose the best general ranker for the product while still preserving row-specific signals: graph for continuation, people/staff for actor/director rows, and metadata for filters and exploration.


## Runtime and Training Progress

The first cell streams the evaluator output while it runs. Classical/product recommenders do not all have a `fit()` step like neural models, so their timing table records the expensive build/evaluation stages: candidate matrix construction, content/graph/people signal construction, SVD factorization, hybrid-weight search, and offline ranking evaluation. Advanced models have a separate training log with the actual learner training time.


In [ ]:
if not classical_training_log.empty:
    classic_times = classical_training_log.copy()
    classic_times['elapsed_minutes'] = (classic_times['elapsed_seconds'] / 60).round(2)
    display(classic_times[['component', 'family', 'status', 'elapsed_seconds', 'elapsed_minutes', 'train_rows', 'eval_rows']])
if not advanced_training_log.empty:
    advanced_times = advanced_training_log.copy()
    advanced_times['elapsed_minutes'] = (advanced_times['elapsed_seconds'] / 60).round(2)
    display(advanced_times.sort_values('elapsed_seconds', ascending=False)[['model', 'family', 'status', 'elapsed_seconds', 'elapsed_minutes', 'train_rows', 'eval_rows', 'note']])
total_classic = classical_training_log['elapsed_seconds'].sum() if not classical_training_log.empty else 0
total_advanced = advanced_training_log['elapsed_seconds'].sum() if not advanced_training_log.empty else 0
pd.DataFrame([
    {'runtime_layer': 'classical/product pipeline', 'seconds': round(total_classic, 1), 'minutes': round(total_classic / 60, 2)},
    {'runtime_layer': 'advanced learned models', 'seconds': round(total_advanced, 1), 'minutes': round(total_advanced / 60, 2)},
])


### Baseline Check

The baseline is the non-personalized popularity ranker. I removed the random row from the final report because it was only a sanity check; the meaningful baseline for this product is whether personalization beats globally popular anime under the same held-out protocol.


In [ ]:
baseline_methods = ['popularity_baseline']
metrics.loc[
    metrics['method'].isin(baseline_methods),
    ['method', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', 'mean_reciprocal_rank', 'median_rank', 'coverage_at_12', 'mean_novelty_at_12']
]

### Classical and Product Rankers

This block compares interpretable rankers before the learned rerankers. These are not just baselines; they are also the building blocks for the final row-based product.

- `metadata_content` answers: "does this match the user's genres, tags, demographics, studios, rating, source, and type?"
- `graph_related` follows explicit anime relation/recommendation edges, so it is strongest for continuations and "because you liked" rows.
- `people_staff_affinity` uses voice actors plus Director, Original Creator, and Original Story signals.
- `item_knn_collaborative` uses item-neighborhood similarity in the collaborative latent space.
- `latent_svd` is the pure collaborative model.
- `full_product_hybrid`, `tuned_product_hybrid`, and `level_tuned_product_hybrid` combine those signals. The best classical/product model is chosen by balanced profile-band Hit@12 first, then overall Hit@12 and MAP@12.


In [ ]:
classical_methods = [
    'metadata_content', 'graph_related', 'people_staff_affinity',
    'item_knn_collaborative', 'latent_svd',
    'full_product_hybrid', 'tuned_product_hybrid', 'level_tuned_product_hybrid',
    'popularity_baseline',
]
classical_table = metrics.loc[
    metrics['method'].isin(classical_methods),
    ['method', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', 'mean_reciprocal_rank', 'median_rank', 'balanced_profile_hit_at_12', 'coverage_at_12', 'mean_novelty_at_12']
].sort_values(['balanced_profile_hit_at_12', 'hit_rate_at_12', 'map_at_12'], ascending=False)
display(classical_table)

best_classical = classical_table.iloc[0]
weight_source = summary['level_tuned_hybrid_weights'] if best_classical['method'] == 'level_tuned_product_hybrid' else summary.get('tuned_hybrid_weights', {})
print(f"Best classical/product ranker: {best_classical['method']} | balanced Hit@12={best_classical['balanced_profile_hit_at_12']:.3f} | overall Hit@12={best_classical['hit_rate_at_12']:.3f}")
if best_classical['method'] == 'level_tuned_product_hybrid':
    for level, weights in weight_source.items():
        print(level, {key: round(float(value), 3) for key, value in weights.items()})
else:
    print({key: round(float(value), 3) for key, value in weight_source.items()})


### Hybrid Weight Search

The fixed SVD/popularity comparison is gone. Hybrid selection now comes from a component search over the actual product signals: SVD, item-KNN, popularity, metadata, graph, and people/staff. The global search picks a single blend; the profile-level search can choose different weights for Beginner, Casual, Fan, and Veteran users.


In [ ]:
if hybrid_component_search.empty:
    print('No global component search artifact found.')
else:
    print('Top global hybrid component blends:')
    component_cols = [col for col in hybrid_component_search.columns if col.startswith('weight_')]
    display(
        hybrid_component_search[
            ['candidate_id', 'evaluated_users', 'balanced_profile_hit_at_12', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', 'mean_reciprocal_rank', *component_cols]
        ].head(12).round(3)
    )

if level_hybrid_component_search.empty:
    print('No profile-level component search artifact found.')
else:
    print('Best blend per user profile band:')
    component_cols = [col for col in level_hybrid_component_search.columns if col.startswith('weight_')]
    best_by_level = (
        level_hybrid_component_search
        .sort_values(['user_level', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12'], ascending=[True, False, False, False])
        .groupby('user_level')
        .head(1)
    )
    display(best_by_level[['user_level', 'candidate_id', 'evaluated_users', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', *component_cols]].round(3))


In [ ]:
metrics

In [ ]:
display(Image(filename=str(PLOT_DIR / 'metric_comparison.png')))
display(Image(filename=str(PLOT_DIR / 'median_rank.png')))
display(Image(filename=str(PLOT_DIR / 'rank_distribution_by_method.png')))


## Advanced Architectures and Learned Rerankers

The advanced comparison asks a second-stage ranking question: once the candidate generators produce possible anime, can a learned model order those candidates better than the tuned hybrid?

These models train on candidate groups from one slice of evaluated users and are tested on a separate slice. They use the same candidate features as the product hybrid, so a win means better ordering of the same available evidence, not a task change.

The current advanced set includes:

- pointwise linear/tree/neural rerankers over product features,
- LightGBM/XGBoost/CatBoost group-aware ranking models,
- implicit ALS collaborative factorization,
- Torch feature MLP and two-tower collaborative models.


In [ ]:
print('Optional advanced architecture inventory:')
with pd.option_context('display.max_colwidth', 260):
    display(advanced_inventory[['architecture', 'family', 'available', 'dependency', 'role']])

print('Model architecture / feature contract:')
with pd.option_context('display.max_colwidth', 260):
    config_cols = ['model', 'family', 'objective', 'architecture_or_layers', 'feature_groups', 'advanced_train_cases', 'advanced_eval_cases']
    display(advanced_model_configs[[col for col in config_cols if col in advanced_model_configs.columns]])

print('Training log and elapsed time:')
display(advanced_training_log.sort_values(['status', 'elapsed_seconds'], ascending=[True, False]) if not advanced_training_log.empty else advanced_training_log)


In [ ]:
if advanced_metrics.empty:
    print('No advanced metrics found. Set RUN_ADVANCED=True and RUN_PIPELINE=True in the first cell.')
else:
    advanced_table = advanced_metrics[
        ['method', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', 'mean_reciprocal_rank', 'median_rank', 'balanced_profile_hit_at_12', 'balanced_profile_map_at_12']
    ].sort_values(['balanced_profile_hit_at_12', 'hit_rate_at_12', 'map_at_12'], ascending=False)
    display(advanced_table)


In [ ]:
advanced_level_metrics


## Discovery-Quality Metrics

Accuracy is not the only product goal. Coverage, novelty, and genre diversity show whether a method is only repeating famous anime or actually expanding the discovery surface. These are interpreted beside Hit@12/MAP@12, because each UI row returns up to 12 titles.


In [ ]:
metrics[['method', 'hit_rate_at_12', 'map_at_12', 'coverage_at_12', 'unique_recommended_at_12', 'mean_novelty_at_12', 'mean_unique_genres_at_12', 'mean_genre_diversity_ratio_at_12']]


In [ ]:
display(Image(filename=str(PLOT_DIR / 'discovery_metrics.png')))


## Results by User Level

The profile-band split is now the main interpretation layer. Beginners and Casual users should not be judged only by raw personalization: popularity and metadata help keep recommendations safe. Fans and Veterans benefit from collaborative signals, graph navigation, and people/staff paths, but they are also harder because many obvious recommendations are already known.

The `full_product_hybrid` is evaluated against every component model so we can see whether blending actually improves the ranking, not just whether SVD beats popularity.


In [ ]:
level_metrics

In [ ]:
display(Image(filename=str(PLOT_DIR / 'user_level_distribution.png')))
display(Image(filename=str(PLOT_DIR / 'hit_at_12_by_user_level.png')))
display(Image(filename=str(PLOT_DIR / 'method_hit_by_user_level.png')))


## Cold Starter and Beginner Entry-Point Candidates

For Cold Starters and Beginners, the system should not immediately behave like a pure collaborative recommender. The generated entry-point list filters out explicit entries and obvious continuation entries, then ranks high-score, high-engagement, shorter or medium-length entry points. This is an onboarding candidate pool, not the final personalized list.

This is the content-based recommendation idea in product form: when behavior is missing, use item metadata and defensible filters rather than pretending the user has a collaborative profile.


In [ ]:
beginner_candidates.head(25)

## Classical/Product Example Recommendation Cases

These examples store held-out titles, the user's training titles, and the tuned product hybrid's top-12 candidate row. They matter because aggregate metrics can hide operational failures: wrong franchise order, side content before parent content, or a VA/staff match that is technically true but too weak to be useful.


In [ ]:
example_top12_col = 'product_hybrid_top12_titles' if 'product_hybrid_top12_titles' in examples.columns else 'hybrid_top12_titles'
example_rank_col = 'level_tuned_hybrid_rank' if 'level_tuned_hybrid_rank' in examples.columns else 'tuned_hybrid_rank'
display_cols = [
    'case_type', 'user_level', 'profile_size', 'holdout_title',
    'popularity_rank', 'latent_svd_rank', example_rank_col,
    'train_titles', example_top12_col,
]
examples[[col for col in display_cols if col in examples.columns]].head(20)


## Advanced Example Recommendation Cases

This table does the same concrete error-analysis work for the best advanced model. It shows the evaluated user, held-out title, rank, model score, and visible top-12 row when available.


In [ ]:
if advanced_error_cases.empty:
    print('No advanced error-case artifact yet. Rerun notebook 08 with RUN_PIPELINE=True.')
else:
    display(advanced_error_cases.head(25))


## Metric Stability and Tuning Decision

The advanced models are now very close to each other. That usually means the feature layer is carrying most of the signal and the learned rerankers are making small ordering improvements. The useful question is not only which metric is highest?, but whether the winner is stable across profile bands and whether it improves early rank quality without hurting coverage/novelty.



In [ ]:
comparison_cols = ['method', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', 'balanced_profile_hit_at_12', 'median_rank', 'mean_rank']
classic_top = metrics.sort_values(['balanced_profile_hit_at_12', 'hit_rate_at_12', 'map_at_12'], ascending=False).head(5)[comparison_cols]
advanced_top = advanced_metrics.sort_values(['balanced_profile_hit_at_12', 'hit_rate_at_12', 'map_at_12'], ascending=False).head(8)[comparison_cols] if not advanced_metrics.empty else pd.DataFrame(columns=comparison_cols)
display(classic_top)
display(advanced_top)
if not advanced_top.empty:
    best_adv = advanced_top.iloc[0]
    best_classic = classic_top.iloc[0]
    deltas = {
        'hit_at_12_gain_vs_best_classic': best_adv['hit_rate_at_12'] - best_classic['hit_rate_at_12'],
        'balanced_hit_gain_vs_best_classic': best_adv['balanced_profile_hit_at_12'] - best_classic['balanced_profile_hit_at_12'],
        'map_at_12_gain_vs_best_classic': best_adv['map_at_12'] - best_classic['map_at_12'],
    }
    display(pd.DataFrame([{'comparison': k, 'delta': round(float(v), 4)} for k, v in deltas.items()]))


## Final Model Choice

The final choice should not be a single universal model. The product is row-based, so each row can use the model family that matches its job.

Decision logic used here:

- pick **one classical/product model** as the transparent fallback and audit anchor;
- pick the strongest **advanced rerankers** for the main general row, preferring models that improve balanced profile Hit@12 without losing MAP@12;
- keep graph/people/content methods as row specialists even if their global held-out metric is lower, because they answer different product questions.

The current run is already strong. Further tuning may improve the third decimal, but the main quality gains now probably come from row guardrails, better candidate generation, and live user-feedback data rather than simply making the offline model bigger.


In [ ]:
classical_rank_cols = ['balanced_profile_hit_at_12', 'hit_rate_at_12', 'map_at_12', 'ndcg_at_12']
classical_sorted = metrics.sort_values(classical_rank_cols, ascending=False).reset_index(drop=True)
best_classical = classical_sorted.head(1).copy()
best_classical['selection_role'] = 'transparent product fallback / general-row guardrail baseline'

advanced_rank_cols = ['balanced_profile_hit_at_12', 'hit_rate_at_12', 'map_at_12', 'ndcg_at_12']
advanced_sorted = advanced_metrics.sort_values(advanced_rank_cols, ascending=False).reset_index(drop=True) if not advanced_metrics.empty else pd.DataFrame()
selected_advanced = advanced_sorted.head(4).copy()
if not selected_advanced.empty:
    selected_advanced['selection_role'] = 'candidate learned reranker for general recommendation row'

final_choices = pd.concat([best_classical, selected_advanced], ignore_index=True, sort=False)
display(final_choices[['method', 'selection_role', 'hit_rate_at_12', 'ndcg_at_12', 'map_at_12', 'balanced_profile_hit_at_12', 'median_rank', 'mean_rank']])

if not best_classical.empty:
    chosen_classic = best_classical.iloc[0]
    print(f"Classical/product choice: {chosen_classic['method']} because it has the best balanced profile Hit@12 among interpretable product hybrids and already includes profile-band-specific weights.")
    if chosen_classic['method'] == 'level_tuned_product_hybrid':
        weights = summary.get('level_tuned_hybrid_weights', {})
        weight_rows = []
        for level, level_weights in weights.items():
            row = {'user_level': level}
            row.update({key: round(float(value), 3) for key, value in level_weights.items()})
            weight_rows.append(row)
        display(pd.DataFrame(weight_rows))

if not selected_advanced.empty:
    top_adv = selected_advanced.iloc[0]
    print(f"Advanced choice for the general row: {top_adv['method']} is currently the strongest balanced-profile learned reranker.")
    print('Keep the next 2-3 advanced models as challengers because the scores are close and future user data may change the ordering.')

print('Product deployment decision: use the best advanced reranker for the general row, level_tuned_product_hybrid as fallback/audit model, graph_related for continuation/because-you-liked rows, and people_staff_affinity for VA/director/creator rows.')
print('Metric interpretation: the current offline setup is strong enough for model selection. More tuning should focus on full-catalog sensitivity, row-specific candidate quality, and future implicit feedback rather than only larger neural layers.')



## Interpretation and Limits

The refreshed evaluation now answers the main feedback points. Popularity remains the explicit baseline. Candidate-pool sensitivity and a full-catalog sample remain in the artifacts. The hybrid layer is justified through global and profile-level weight searches, and concrete error-case tables connect evaluated users, held-out items, ranks, and model scores.

The learned rerankers are evaluated on a separate train/eval slice of candidate groups. They use the same candidate features as the product hybrid, so the comparison is fair: if a learned model wins, it is improving the ordering of the same signals, not changing the task.

Dropped/negative ratings are not treated as positive seeds. The main positive layer is scored catalog-matched ratings >= 7, excluding dropped rows when status exists. Dropped and low-score titles are still useful for product guardrails and future explicit negative modeling, but they are not counted as liked interactions in this offline hit-rate protocol.


## Reproducibility

Run the recommender evaluation pipeline from the repository root with:

```bash
python src/08_evaluate_recommenders.py --min-item-positives 10 --advanced-train-cases 8000 --advanced-eval-cases 10000
```

To use every eligible eval user, omit `--max-eval-users`. To test a faster smoke run, add for example `--max-eval-users 5000`. The advanced-model cells show training time, architecture, and the selected hybrid weights so model decisions are auditable from inside the notebook.

